In [ ]:
import re
import math
from collections import Counter

def chunk_by_section(document_text):
    return re.split(r"\n## ", document_text)

In [ ]:
class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.documents, self._corpus_tokens, self._doc_len = [], [], []
        self._doc_freqs, self._idf = {}, {}
        self._avg_doc_len = 0.0
        self._index_built = False
        self.k1, self.b = k1, b

    def _tokenize(self, text):
        return [t for t in re.split(r"\W+", text.lower()) if t]

    def add_document(self, document):
        tokens = self._tokenize(document["content"])
        self.documents.append(document)
        self._corpus_tokens.append(tokens)
        self._doc_len.append(len(tokens))
        seen = set()
        for t in tokens:
            if t not in seen:
                self._doc_freqs[t] = self._doc_freqs.get(t, 0) + 1
                seen.add(t)
        self._index_built = False

    def _build_index(self):
        N = len(self.documents)
        self._avg_doc_len = sum(self._doc_len) / N if N else 0
        self._idf = {term: math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
                     for term, freq in self._doc_freqs.items()}
        self._index_built = True

    def _score(self, query_tokens, doc_index):
        score = 0.0
        counts = Counter(self._corpus_tokens[doc_index])
        dl = self._doc_len[doc_index]
        for t in query_tokens:
            if t not in self._idf: continue
            tf = counts.get(t, 0)
            score += (self._idf[t] * tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * (dl / self._avg_doc_len)) + 1e-9)
        return score

    def search(self, query_text, k=1, norm_factor=0.1):
        if not self._index_built: self._build_index()
        tokens = self._tokenize(query_text)
        raw = [(self._score(tokens, i), self.documents[i]) for i in range(len(self.documents))]
        raw = [(s, d) for s, d in raw if s > 1e-9]
        raw.sort(key=lambda x: x[0], reverse=True)
        results = [(d, math.exp(-norm_factor * s)) for s, d in raw[:k]]
        results.sort(key=lambda x: x[1])
        return results

In [ ]:
# Load, chunk, index
with open("report.md", "r") as f:
    text = f.read()
chunks = chunk_by_section(text)

store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})
print(f"BM25 index: {len(store.documents)} documents")

In [ ]:
# Search for specific incident ID
results = store.search("What happened with INC-2023-Q4-011?", k=3)
for doc, dist in results:
    print(f"  {dist:.4f}: {doc['content'].strip().split(chr(10))[0][:80]}")

In [ ]:
# Search for medical term
results = store.search("Tell me about XDR-471 syndrome", k=3)
for doc, dist in results:
    print(f"  {dist:.4f}: {doc['content'].strip().split(chr(10))[0][:80]}")